# Inspect Runoff Target

QA/QC of the combined runoff calibration target written by
`targets/run.py` to `<project>/targets/runoff_targets.nc` (and the
optional `runoff_targets_nn_filled.nc` companion).

The runoff target is a per-HRU per-month **`(lower_bound, upper_bound)`
range in cfs**, formed as the NaN-aware min / max across three sources
on a shared fabric:

- ERA5-Land `ro` → mm/month → cfs (recipes §1).
- GLDAS-2.1 NOAH `runoff_total` → mm/month → cfs.
- MWBM (ClimGrid, `runoff`) → mm/month → cfs.

The accompanying `n_sources` flag (0 / 1 / 2 / 3) records how many
sources contributed at each cell — the bound is NaN only when
`n_sources == 0`. The `nn_filled` companion file reuses the bounds
but additionally fills NaN HRUs from up to 10 nearest neighbours; the
`nn_filled` int8 flag (0 / 1) marks which HRU-times were filled.

This notebook checks:

- File schema and global metadata (period, fabric SHA, source string).
- Per-time `n_sources` coverage and where the gaps fall geographically.
- `lower_bound`, `upper_bound`, and range size at a representative time.
- CONUS area-weighted mean lower / upper time series.
- Representative-HRU time series across the four physiographic regimes.
- NN-fill: which HRU-times were filled and whether the filled values
  preserve the lower / upper distribution.
- Order-of-magnitude sanity check against published CONUS runoff.

Companion to:

- `inspect_consolidated_runoff.ipynb` — pre-aggregation gridded NCs.
- `inspect_aggregated_runoff.ipynb` — per-source HRU aggregates before
  multi-source combination.


## Conventions

- HRU dim name follows `fabric.id_col` from the project config
  (e.g. `nat_hru_id` for the GFv2 fabric, `nhm_id` elsewhere).
- Units are read from the **target NC variable attrs** (which are
  authoritative for the combined target — the catalog only documents
  per-source units, not the post-combination cfs).
- `TARGET_TIME` (set below) drives the at-time choropleth panels.
  Pick a month with `n_sources == 3` everywhere if you want the
  cleanest snapshot; an early-period or late-period month exposes the
  partial-coverage corners.
- All maps are CONUS-Albers-equivalent only inside `area_weighted_mean`
  / `area_weighted_series` — the fabric is plotted in EPSG:4326 because
  the cost of reprojecting 360k polygons on every map is not worth it
  at inspection time.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    area_weighted_series,
    discover_target_nc,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    n_sources_per_time,
    nan_hru_count,
    open_target_nc,
    plot_categorical_choropleth,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/targets/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

TARGET = "runoff"
TARGET_YEAR = 2005
TARGET_MONTH = 4  # April -- snowmelt-driven peak, n_sources == 3 everywhere
TARGET_TIME = f"{TARGET_YEAR}-{TARGET_MONTH:02d}-01"

REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)
id_dim = fabric_cfg["id_col"]

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs, id_col={id_dim!r})")
print(f"Target time: {TARGET_TIME}")


## Open and summarise the target NCs

`discover_target_nc` finds both the unfilled (`runoff_targets.nc`) and
NN-filled (`runoff_targets_nn_filled.nc`) variants if they exist.
Either may be `None` — this cell skips with a clear message and
downstream cells iterate only over what was loaded.


In [ ]:
raw_path, filled_path = discover_target_nc(project_dir, TARGET)

if raw_path is None:
    print(f"SKIP: {TARGET}_targets.nc not found at {project_dir / 'targets'}.")
    print("      Run `pixi run run-runoff -- --project-dir <project>` first.")
    raise SystemExit

ds_raw = open_target_nc(raw_path)
print(f"Loaded raw target:    {raw_path.name}  ({raw_path.stat().st_size / 1e6:.1f} MB)")

ds_filled = None
if filled_path is not None:
    ds_filled = open_target_nc(filled_path)
    print(
        f"Loaded NN-filled:     {filled_path.name}  "
        f"({filled_path.stat().st_size / 1e6:.1f} MB)"
    )
else:
    print("NN-filled variant absent (set `nn_fill: true` in config to produce it).")


## Schema and global metadata

The global attrs carry the provenance you'd want at calibration time:
the period, the fabric SHA-256 (so a downstream consumer can detect a
fabric swap), the comma-separated `source` string, and the references
to TM 6-B10. Per-variable attrs carry the units (`cfs` for the bounds,
`1` for the diagnostic flags).


In [ ]:
print(ds_raw)
print()
print("=== Global attrs ===")
for k, v in ds_raw.attrs.items():
    print(f"  {k:<20} {v}")

print()
print("=== Per-variable units ===")
for v in ("lower_bound", "upper_bound", "n_sources"):
    units = ds_raw[v].attrs.get("units", "(no units attr)")
    long_name = ds_raw[v].attrs.get("long_name", "")
    print(f"  {v:<14} units={units!r}  long_name={long_name!r}")


## Per-time coverage

`n_sources_per_time` returns the per-month count of HRUs at each flag
value (0 / 1 / 2 / 3 for a 3-source target). Two diagnostics:

- The `n=0` column is the count of all-NaN HRUs at that timestep —
  these are the cells the NN-fill targets.
- A jump in `n=2` (or drop in `n=3`) at a particular month flags a
  source whose period ended there. For runoff: MWBM ClimGrid ends
  2014-12; ERA5-Land continues; GLDAS-2.1 ends 2014-12 too. So
  inside the 2000–2010 default window, all three should contribute
  almost everywhere.


In [ ]:
cov = n_sources_per_time(ds_raw)
print(cov.describe().T[["mean", "std", "min", "max"]])

fig, ax = plt.subplots(figsize=(11, 4))
cov.plot(ax=ax)
ax.set_xlabel("Time")
ax.set_ylabel("HRU count")
ax.set_title(f"Per-month n_sources distribution — {TARGET} target")
ax.legend(title="flag value", loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_coverage_timeseries")
plt.show()


## n_sources map at TARGET_TIME

Categorical choropleth — colours match the `flag_values` attr on the
`n_sources` variable. NaN HRUs (where the time slice is missing) plot
in light grey; the legend reports the count of HRUs at each flag value
so coverage anomalies are visible at a glance.


In [ ]:
ns = select_month(ds_raw["n_sources"], TARGET_YEAR, TARGET_MONTH).to_pandas()

categories = {
    0: ("0 sources (all NaN)", "crimson"),
    1: ("1 source", "khaki"),
    2: ("2 sources", "lightgreen"),
    3: ("3 sources", "darkgreen"),
}

fig, ax = plt.subplots(figsize=(11, 7))
plot_categorical_choropleth(
    ax,
    fabric,
    ns,
    categories=categories,
    title=f"n_sources at {TARGET_TIME} — {TARGET} target",
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_n_sources_map")
plt.show()


## Lower / upper / range maps at TARGET_TIME

Three side-by-side panels:

- `lower_bound` — NaN-aware min across the three sources (cfs).
- `upper_bound` — NaN-aware max across the three sources (cfs).
- `range = upper - lower` — the per-HRU per-month spread the NHM
  calibrator is asked to land inside. Wide ranges = poor cross-source
  agreement; tight ranges = consensus among the LSMs at that cell.

Colour scale is anchored on the 2nd / 98th percentile of `upper_bound`
so the long tail of large-area HRUs (which dominate cfs because cfs is
volume-rate-per-HRU, not depth) doesn't flatten the rest of the map.


In [ ]:
lb = select_month(ds_raw["lower_bound"], TARGET_YEAR, TARGET_MONTH).to_pandas()
ub = select_month(ds_raw["upper_bound"], TARGET_YEAR, TARGET_MONTH).to_pandas()
rng = ub - lb

ub_finite = ub.dropna().values
vmin = float(np.percentile(ub_finite, 2))
vmax = float(np.percentile(ub_finite, 98))

units = ds_raw["lower_bound"].attrs.get("units", "cfs")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
plot_hru_choropleth(
    axes[0], fabric, lb, vmin=vmin, vmax=vmax, cmap="YlGnBu",
    title=f"lower_bound\n{TARGET_TIME} | {units}", units=units,
)
plot_hru_choropleth(
    axes[1], fabric, ub, vmin=vmin, vmax=vmax, cmap="YlGnBu",
    title=f"upper_bound\n{TARGET_TIME} | {units}", units=units,
)
plot_hru_choropleth(
    axes[2], fabric, rng, vmin=0, vmax=vmax, cmap="OrRd",
    title=f"range = upper - lower\n{TARGET_TIME} | {units}", units=units,
)
fig.suptitle(f"{TARGET} target bounds — {TARGET_TIME}", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_bounds_map")
plt.show()

print(f"lower CONUS area-weighted mean: {area_weighted_mean(lb, fabric):,.2f} {units}")
print(f"upper CONUS area-weighted mean: {area_weighted_mean(ub, fabric):,.2f} {units}")
print(f"range CONUS area-weighted mean: {area_weighted_mean(rng, fabric):,.2f} {units}")


## NaN HRU coverage

HRUs where `n_sources == 0` (and therefore `lower_bound` / `upper_bound`
are NaN). These are the cells that nearest-neighbour fill targets — at
this time slice they show as red. A handful of NaN HRUs is normal for
the 2000–2010 window (typically very small fabric polygons whose source
grids didn't intersect at all); a large red blob means a source's
period or footprint isn't covering what we expect.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
plot_nan_hrus(
    ax, fabric, lb,
    title=(
        f"NaN HRUs (red) — {TARGET_TIME} | "
        f"{nan_hru_count(lb)} of {len(fabric)} "
        f"({100 * nan_hru_count(lb) / len(fabric):.2f}%)"
    ),
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_nan_map")
plt.show()


## CONUS area-weighted lower / upper time series

`area_weighted_series` reduces the (time, hru) bound arrays to a
per-month CONUS-mean, weighted by EPSG:5070 polygon area. Lower /
upper plotted together form an envelope — its width is the average
cross-source disagreement nationally.

The seasonal cycle is the dominant signal — peak in spring (snowmelt)
and a secondary autumn rise in the Pacific Northwest / Northeast.
Watch for:

- A flat lower bound near zero in summer (drought regimes).
- An expansion of the envelope in spring (snowmelt brings out
  ERA5-Land vs GLDAS vs MWBM physics differences).
- Step changes when a source's period boundary crosses the time axis.


In [ ]:
lb_series = area_weighted_series(ds_raw["lower_bound"], fabric, id_dim)
ub_series = area_weighted_series(ds_raw["upper_bound"], fabric, id_dim)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(
    lb_series.index, lb_series.values, ub_series.values,
    color="steelblue", alpha=0.25, label="lower–upper envelope",
)
ax.plot(lb_series.index, lb_series.values, color="steelblue", lw=1, label="lower")
ax.plot(ub_series.index, ub_series.values, color="darkblue", lw=1, label="upper")
ax.set_xlabel("Time")
ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
ax.set_title(f"{TARGET} target — CONUS area-weighted bounds")
ax.legend()
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_conus_series")
plt.show()


## Representative HRU time series

Four physiographically distinct points; `lookup_hrus_by_points` uses
`gpd.sjoin` to resolve each (lon, lat) to the containing HRU. The
lower / upper envelope at each point is the per-month spread the
calibrator targets locally — narrower envelopes = stronger inter-source
agreement at that cell.


In [ ]:
rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
print("Representative HRUs:", rep_hrus)

lb_at = ds_raw["lower_bound"].sel({id_dim: list(rep_hrus.values())})
ub_at = ds_raw["upper_bound"].sel({id_dim: list(rep_hrus.values())})

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
    lb_h = lb_at.sel({id_dim: hru_id}).values
    ub_h = ub_at.sel({id_dim: hru_id}).values
    t = pd.DatetimeIndex(lb_at.time.values)
    ax.fill_between(t, lb_h, ub_h, color="steelblue", alpha=0.25)
    ax.plot(t, lb_h, color="steelblue", lw=1, label="lower")
    ax.plot(t, ub_h, color="darkblue", lw=1, label="upper")
    ax.set_title(f"{label} (HRU {hru_id})")
    ax.set_ylabel(f"runoff ({units})")
    ax.legend(fontsize=8)
fig.suptitle(f"{TARGET} target at representative HRUs", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_representative_series")
plt.show()


## NN-fill comparison

The companion `runoff_targets_nn_filled.nc` reuses the same bounds but
fills NaN HRUs from up to `nn_max_candidates=10` nearest neighbours
(see `targets/run.py`). The `nn_filled` int8 flag marks each
HRU-time as `0` (not filled) or `1` (filled).

Two checks:

- **Filled-flag map at TARGET_TIME** — should match the NaN-HRU map
  one-for-one (any NaN that *can* be filled gets a `1`).
- **Distribution preservation** — the area-weighted CONUS mean of the
  filled bounds should track the unfilled mean closely (the NN fill
  is interpolating from immediate neighbours, not synthesising). A
  large jump indicates an over-aggressive fill.


In [ ]:
if ds_filled is None:
    print("SKIP: NN-filled variant not present.")
else:
    flag = select_month(ds_filled["nn_filled"], TARGET_YEAR, TARGET_MONTH).to_pandas()
    n_filled = int((flag == 1).sum())
    print(
        f"NN-filled at {TARGET_TIME}: {n_filled} HRUs filled "
        f"({100 * n_filled / len(fabric):.2f}%)"
    )

    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    plot_categorical_choropleth(
        axes[0], fabric, flag,
        categories={0: ("not filled", "lightgrey"), 1: ("filled", "crimson")},
        title=f"nn_filled flag at {TARGET_TIME}",
    )
    lb_f = select_month(ds_filled["lower_bound"], TARGET_YEAR, TARGET_MONTH).to_pandas()
    diff = (lb_f - lb).abs().fillna(lb_f)  # NaN in raw -> show filled value
    plot_hru_choropleth(
        axes[1], fabric, diff, vmin=0, vmax=vmax / 4, cmap="OrRd",
        title=f"|lower_bound_filled - lower_bound_raw| at {TARGET_TIME}",
        units=units,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_map")
    plt.show()

    lb_f_series = area_weighted_series(ds_filled["lower_bound"], fabric, id_dim)
    ub_f_series = area_weighted_series(ds_filled["upper_bound"], fabric, id_dim)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(lb_series.index, lb_series.values, color="steelblue", lw=1, label="lower (raw)")
    ax.plot(lb_f_series.index, lb_f_series.values, color="steelblue", lw=1, ls="--", label="lower (filled)")
    ax.plot(ub_series.index, ub_series.values, color="darkblue", lw=1, label="upper (raw)")
    ax.plot(ub_f_series.index, ub_f_series.values, color="darkblue", lw=1, ls="--", label="upper (filled)")
    ax.set_xlabel("Time")
    ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
    ax.set_title(f"{TARGET} target — raw vs NN-filled CONUS series")
    ax.legend()
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_series")
    plt.show()


## Order-of-magnitude sanity check

Sum the per-HRU `lower_bound` and `upper_bound` (cfs) across the
fabric at `TARGET_TIME` to get total CONUS surface-runoff generation
that month. Compare to a published reference: USGS WaterWatch /
TM 6-B10 reports a CONUS-mean **annual** runoff of ~24 in/yr, which
maps to a long-term-mean total CONUS runoff rate of
~1.9×10⁵ m³/s ≈ ~6.7×10⁶ cfs. Monthly values vary by season (peak in
spring, trough in late summer) — expect the snapshot to land within
roughly 0.5×–2× of the annual mean.

A summed lower-bound that is *orders of magnitude* off the reference
is a smoking gun for a missed conversion factor (per
`feedback_validate_magnitudes.md`).


In [ ]:
lb_total = float(np.nansum(lb.values))
ub_total = float(np.nansum(ub.values))
ref_cfs = 6.7e6  # CONUS-mean annual runoff (~24 in/yr) expressed as cfs

print(f"CONUS-summed lower_bound at {TARGET_TIME}: {lb_total:>16,.0f} cfs")
print(f"CONUS-summed upper_bound at {TARGET_TIME}: {ub_total:>16,.0f} cfs")
print(f"Reference (CONUS annual-mean runoff):     {ref_cfs:>16,.0f} cfs")
print(
    f"Ratio (lower / ref):                          {lb_total / ref_cfs:>14.3f}x "
    f"(expect 0.3..3.0)"
)
print(
    f"Ratio (upper / ref):                          {ub_total / ref_cfs:>14.3f}x "
    f"(expect 0.5..3.0)"
)


In [ ]:
ds_raw.close()
if ds_filled is not None:
    ds_filled.close()
